# ⚖️ Day 1 미니 프로젝트 — 법률 Q&A 챗봇 v0.1 (EXAONE 완성판)

**모델:** `LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct` (로컬 추론)

---

## 📋 오늘 만들 챗봇 구조

```
legal_chatbot()              ← 메인 함수
    ├── system_prompt        ← AI 역할·제약 설정
    ├── history              ← 대화 기록 리스트
    └── while True:          ← 대화 루프
            └── chat_with_history()  ← 멀티턴 로컬 추론
                    └── safe_generate()  ← 안전한 호출
                            └── generate_response()  ← EXAONE 추론
```

## ⏱️ 시간 계획

```
16:00–16:10  환경 설정 + 모델 로드 (EXAONE 다운로드 약 2~3분)
16:10–16:30  Step 1~4 완성 코드 실행
16:30–16:40  챗봇 테스트
16:40–16:50  품질 불일치 수집 (Day 2 입력 데이터)
16:50–17:00  저장 + 시스템 프롬프트 변경 실험
```

---
## 0. 환경 설정 — GPU 확인 + EXAONE 모델 로드

In [1]:
# GPU 확인
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Tesla T4, 15360 MiB


In [2]:
# 패키지 설치
!pip install -q --upgrade transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 55.7 MB/s eta 0:00:00


In [4]:
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("CUDA 빌드:", torch.version.cuda)
print("TorchVision:", torchvision.__version__)
print("PyTorch 경로:", torch.__file__)
print("TorchVision 경로:", torchvision.__file__)
print("GPU 사용 가능:", torch.cuda.is_available())

from torchvision.extension import _has_ops
print("TorchVision 연산 사용 가능:", _has_ops())

PyTorch: 2.8.0+cu128
CUDA 빌드: 12.8
TorchVision: 0.23.0+cu128
PyTorch 경로: /workspace/.venv/lib/python3.12/site-packages/torch/__init__.py
TorchVision 경로: /workspace/.venv/lib/python3.12/site-packages/torchvision/__init__.py
GPU 사용 가능: True
TorchVision 연산 사용 가능: True


In [5]:
# 토크나이저 로드
import os
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
model_name = "LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct"

print("📥 토크나이저 로드 중...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
print("✅ 토크나이저 로드 완료!")

📥 토크나이저 로드 중...


✅ 토크나이저 로드 완료!


In [ ]:
# 모델 로드 (약 2~3분 소요)
# ⚠️ ReadTimeout 시 이 셀만 다시 실행하세요

print("📥 모델 로드 중... (약 2~3분 소요)")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="auto",
)
print(f"✅ 모델 로드 완료! GPU 메모리: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

📥 모델 로드 중... (약 2~3분 소요)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


modeling_exaone.py:   0%|          | 0.00/24.0k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct:
- modeling_exaone.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] The `check_model_inputs` decorator is deprecated in favor of `merge_with_config_defaults`.


[ERROR] `cache_position` is part of ExaoneModel.forward's signature, but not documented. Make sure to add it to the docstring of the function in /root/.cache/huggingface/modules/transformers_modules/LGAI_hyphen_EXAONE/EXAONE_hyphen_3_dot_5_hyphen_2_dot_4B_hyphen_Instruct/ccce25bd39c141fe053e0bc75818a8f5fe962802/modeling_exaone.py.
[ERROR] `cache_position` is part of ExaoneForCausalLM.forward's signature, but not documented. Make sure to add it to the docstring of the function in /root/.cache/huggingface/modules/transformers_modules/LGAI_hyphen_EXAONE/EXAONE_hyphen_3_dot_5_hyphen_2_dot_4B_hyphen_Instruct/ccce25bd39c141fe053e0bc75818a8f5fe962802/modeling_exaone.py.


model.safetensors.index.json:   0%|          | 0.00/22.2k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
# 추론 유틸리티 함수 + 토큰 추적기
import time

def generate_response(messages, max_new_tokens=300, temperature=0.3, top_p=0.9):
    """EXAONE 로컬 추론 — 메시지 리스트를 받아 응답 dict를 반환한다."""
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    input_ids = tokenizer(
        prompt, return_tensors="pt", add_special_tokens=False
    ).input_ids.to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True if temperature > 0 else False,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][input_ids.shape[1]:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return {
        "content": answer,
        "input_tokens": input_ids.shape[1],
        "output_tokens": len(generated_ids),
    }


class TokenTracker:
    """로컬 추론의 누적 토큰 수를 추적한다."""
    def __init__(self):
        self.total_in = 0
        self.total_out = 0
        self.calls = 0

    def add(self, result):
        self.total_in += result["input_tokens"]
        self.total_out += result["output_tokens"]
        self.calls += 1

    def report(self):
        # API 환산 비용 (gpt-4o-mini 기준 비교용)
        api_cost_usd = (self.total_in * 0.15 + self.total_out * 0.60) / 1_000_000
        print(f"\n📊 {self.calls}회 | 입력:{self.total_in:,} | 출력:{self.total_out:,} 토큰")
        print(f"   💰 로컬 추론 비용: ₩0")
        print(f"   📌 (참고) API 환산 비용: ${api_cost_usd:.4f} (약 {api_cost_usd*1400:.1f}원)")

tracker = TokenTracker()
print("✅ 환경 설정 완료")

---
## Step 1 | 시스템 프롬프트 설계

법률 AI의 **역할 · 규칙 · 금지 사항**을 설정합니다.

| 요소 | 설명 | 예시 |
|------|------|------|
| 역할 | AI가 누구인지 | `"당신은 법률 전문 AI입니다."` |
| 규칙 | 어떻게 답변할지 | `"3문장 이내로"` |
| 금지 | 하면 안 되는 것 | `"법조문 번호는 제공하지 않음"` |

In [ ]:
domain        = "법률"
max_sentences = 3

system_prompt = f"""당신은 {domain} 전문 AI 어시스턴트입니다.

다음 규칙을 반드시 지키세요:
- 답변은 {max_sentences}문장 이내로 간결하게
- 불확실한 경우 '전문 변호사 상담 권장' 명시
- 판례나 법조문 번호는 제공하지 않음
- 개인 식별 정보는 수집하지 않음
""".strip()

print("📋 시스템 프롬프트:")
print(system_prompt)

---
## Step 2 | safe_generate 함수

로컬 추론이 실패해도 멈추지 않고 자동으로 재시도하는 함수입니다.

> ⚠️ **API와 로컬의 차이**: API는 네트워크/Rate Limit 오류가 주원인이라 지수 백오프(1→2→4초)가 효과적이지만, 로컬은 GPU 메모리 오류·토큰 초과가 주원인이므로 **GPU 캐시 정리 후 즉시 재시도**가 더 효과적입니다.

In [ ]:
def safe_generate(messages, temperature=0.3, max_new_tokens=300, max_retries=2):
    """로컬 추론 실패 시 자동으로 재시도한다."""

    for attempt in range(max_retries):
        try:
            result = generate_response(
                messages,
                temperature=temperature,
                max_new_tokens=max_new_tokens,
            )
            tracker.add(result)
            return result["content"]

        except Exception as e:
            print(f"  ⚠️ [{attempt+1}/{max_retries}] 오류: {e}")
            if attempt < max_retries - 1:
                print(f"     → GPU 메모리 정리 후 재시도...")
                torch.cuda.empty_cache()

    print("  ❌ 최대 재시도 초과")
    return None

In [ ]:
# 🔬 테스트
result = safe_generate(
    [{"role": "user", "content": "안녕하세요, 한 줄로 인사해 주세요."}]
)

if result:
    print(f"✅ 성공! 응답: {result}")
else:
    print("❌ None이 반환됨")

---
## Step 3 | chat_with_history 함수

대화 기록을 유지하면서 멀티턴으로 대화하는 함수입니다.

```
동작 순서:
  1. user 메시지를 history에 추가
  2. history 전체를 모델에 전달  ← 이래야 맥락이 유지됩니다!
  3. 실패하면 → user 메시지 제거 (history 보호)
  4. 성공하면 → assistant 응답을 history에 추가
```

In [ ]:
def chat_with_history(history, user_input, temperature=0.3, max_new_tokens=300):
    """대화 기록을 유지하면서 EXAONE을 호출한다."""

    # 1단계: user 메시지 추가
    history.append({"role": "user", "content": user_input})

    # 2단계: 추론 호출 (history 전체를 보내야 맥락 유지!)
    result = safe_generate(
        history,
        temperature=temperature,
        max_new_tokens=max_new_tokens,
    )

    # 3단계: 실패 처리 — history 보호
    if result is None:
        history.pop()   # 방금 추가한 user 메시지 제거
        return "⚠️ 응답을 받지 못했습니다. 다시 시도해 주세요."

    # 4단계: assistant 응답 추가
    history.append({"role": "assistant", "content": result})

    return result

In [ ]:
# 🔬 테스트: 2턴 대화로 맥락 확인
test_history = [{"role": "system", "content": system_prompt}]

print("[1턴] 계약서에서 가장 중요한 조항은?")
ans1 = chat_with_history(test_history, "계약서에서 가장 중요한 조항은?")
print(f"답변: {ans1}")

print()
print("[2턴] 방금 말한 내용을 요약해 주세요.")
ans2 = chat_with_history(test_history, "방금 말한 내용을 요약해 주세요.")
print(f"답변: {ans2}")

print()
print(f"history 길이: {len(test_history)} (예상: 5 = system + 2턴×2)")

if len(test_history) == 5:
    print("✅ 멀티턴 정상 동작! 2번째 답변이 1번째 내용을 참조하나요?")
else:
    print("❌ 길이가 맞지 않음")

---
## Step 4 | legal_chatbot 메인 함수

while 루프로 사용자와 계속 대화하는 챗봇입니다.

In [ ]:
def legal_chatbot():
    """법률 Q&A 챗봇 v0.1 (EXAONE)"""

    # 히스토리 초기화
    history = [{"role": "system", "content": system_prompt}]

    print("=" * 50)
    print("⚖️ 법률 Q&A 챗봇 v0.1 (EXAONE 3.5)")
    print("  종료: quit 또는 q 입력")
    print("=" * 50)

    while True:

        user_input = input("\n❓ 질문: ").strip()

        # 종료
        if user_input.lower() in ["quit", "q"]:
            print("\n👋 챗봇을 종료합니다.")
            break

        # 빈 입력
        if not user_input:
            print("  질문을 입력해 주세요.")
            continue

        # 추론 호출
        answer = chat_with_history(history, user_input)

        print(f"\n💬 답변: {answer}")

    turns = (len(history) - 1) // 2
    print(f"\n📊 총 {turns}턴 대화 완료")
    tracker.report()
    return history

---
## Step 5 | 챗봇 실행!

아래 순서로 질문을 입력해보세요.

| 순서 | 입력 | 확인할 것 |
|------|------|----------|
| 1 | `임차인이 보증금을 돌려받지 못하면 어떻게 해야 하나요?` | 3문장 이내? |
| 2 | `방금 말한 방법 중 가장 빠른 게 뭔가요?` | 1번 답변 참조? |
| 3 | `그 절차를 더 자세히 설명해 주세요.` | 맥락 연결? |
| 4 | `민법 제9999조를 설명해 주세요.` | 없는 조문 안 만듦? |
| 5 | `quit` | 정상 종료? |

In [ ]:
# 챗봇 실행!
chat_history = legal_chatbot()

In [ ]:
# 결과 확인
turns = (len(chat_history) - 1) // 2
print(f"총 대화: {turns}턴")
print(f"role 순서: {[m['role'] for m in chat_history]}")
print()

checks = [
    (chat_history[0]["role"] == "system",   "system 프롬프트 정상"),
    (turns >= 1,                             "1턴 이상 대화함"),
    ("assistant" in [m["role"] for m in chat_history], "AI 응답 기록됨"),
]
for ok, msg in checks:
    print(f"  {'✅' if ok else '❌'} {msg}")

---
## Step 6 | 품질 불일치 수집 ⚠️ 필수!

> **Day 2 오전 실습에서 이 데이터를 사용합니다!**

같은 질문을 3번 보내서 답변이 어떻게 다른지 관찰합니다.

In [ ]:
# 질문 선택 (0, 1, 2 중 하나 또는 직접 작성)
questions_pool = [
    "임차인이 보증금을 돌려받지 못할 때 법적으로 어떤 조치를 취할 수 있나요?",
    "근로계약서 없이 일한 경우 임금 체불 시 어떻게 대처해야 하나요?",
    "온라인 쇼핑 환불 거부 시 소비자 권리는 무엇인가요?",
]

my_question = questions_pool[0]   # ← 원하는 번호로 바꿔보세요

print(f"질문: {my_question}")
print(f"조건: temperature=0.3, 3회 반복")
print()

samples = []

for i in range(3):
    result = generate_response(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": my_question}
        ],
        temperature=0.3,
        max_new_tokens=300,
    )
    answer = result["content"]
    samples.append(answer)
    tracker.add(result)

    print(f"{'─'*50}")
    print(f"[{i+1}회차] ({len(answer)}자, {result['output_tokens']}토큰)")
    print(answer)
    print()

In [ ]:
# 자동 분석
print("="*50)
print("📋 차이 분석")
print()

for i, ans in enumerate(samples):
    has_law  = any(kw in ans for kw in ["법", "조", "민법", "임대차"])
    has_list = any(c  in ans for c  in ["1.", "①", "첫째", "- "])
    has_tip  = any(kw in ans for kw in ["상담", "변호사", "전문"])

    print(f"[{i+1}회차] {len(ans)}자 | 법률용어:{'✅' if has_law else '❌'} | 구조화:{'✅' if has_list else '❌'} | 상담권장:{'✅' if has_tip else '❌'}")

print()
if len(set(samples)) == 1:
    print("→ 3회 모두 동일")
else:
    print("→ ⚠️ 답변이 다릅니다! 이게 '품질 불일치' 현상입니다.")
    print()
    print("💡 내일 Day 2에서 Zero-shot / Few-shot / CoT로 이 불일치를 개선합니다!")

In [ ]:
# 기록 노트 (___ 부분을 직접 채우세요)
quality_note = {
    "question":    my_question,
    "model":       "EXAONE-3.5-2.4B-Instruct",
    "temperature": 0.3,
    "samples":     samples,
    "my_observation": {
        "1회차": {"구조": "___", "정확도": "___", "특이사항": "___"},
        "2회차": {"구조": "___", "정확도": "___", "특이사항": "___"},
        "3회차": {"구조": "___", "정확도": "___", "특이사항": "___"},
        "가장_큰_차이": "___",  # 예: "1회차는 목록형, 3회차는 서술형"
        "day2_가설":   "___",  # 예: "Few-shot을 쓰면 형식이 통일될 것 같다"
    }
}
print("📝 기록 노트 준비 완료. ___ 부분을 실제 관찰 내용으로 바꿔주세요.")

---
## Step 7 | 저장 (Drive 또는 로컬)

In [ ]:
# Drive 마운트 시도 (실패하면 로컬에 저장)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    save_dir = "/content/drive/MyDrive/llm_course/day1"
except Exception as e:
    print(f"⚠️ Drive 마운트 실패: {e}")
    save_dir = "/content"

os.makedirs(save_dir, exist_ok=True)

# 챗봇 대화 기록
with open(f"{save_dir}/chat_history.json", "w", encoding="utf-8") as f:
    json.dump(chat_history, f, ensure_ascii=False, indent=2)
print(f"✅ chat_history.json 저장: {save_dir}")

# 품질 불일치 노트 ← Day 2에서 사용!
with open(f"{save_dir}/quality_note.json", "w", encoding="utf-8") as f:
    json.dump(quality_note, f, ensure_ascii=False, indent=2)
print(f"✅ quality_note.json 저장: {save_dir}")

print()
print("📌 내일 quality_note.json을 불러와 Day 2 실습을 시작합니다!")
print()
tracker.report()

---
## Step 8 | 시스템 프롬프트 변경 실험 (여유 시간)

코드는 그대로, **system_prompt만 바꿔서** 어떻게 달라지는지 확인합니다.

In [ ]:
# 실험 A: 제약 제거 → 환각 발생 확인
h_a = [{"role": "system", "content": "당신은 법률 전문 AI 어시스턴트입니다."}]
r_a = chat_with_history(h_a, "민법 제999조를 설명해 주세요.")

print("[제약 없음]")
print(r_a)
print()
print("→ 없는 조문을 만들어낼 가능성이 높습니다 (환각)")

제999조 (상속회복청구권)①상속권이 참칭상속권자로 인하여 침해된 때에는 상속권자 또는 그 법정대리인은 상속회복의 소를 제기할 수 있다.

②제1항의 상속회복청구권은 그 침해를 안 날부터 3년, 상속권의 침해행위가 있은 날부터 10년을 경과하면 소멸된다.  <개정 2002. 1. 14.>

[전문개정 1990. 1. 13.]

In [ ]:
# 실험 B: 도메인 변경 (법률 → 의료)
h_b = [{"role": "system", "content": """당신은 내과 전문의 AI입니다.
- 3문장 이내로 답변
- 불확실하면 '전문 의료진 상담 권장' 명시
- 특정 약물 용량은 제시하지 않음"""}]
r_b = chat_with_history(h_b, "두통이 자주 생기는데 원인이 뭘까요?")

print("[의료 도메인]")
print(r_b)
print()
print("💡 코드 한 줄 바꾸지 않고 완전히 다른 챗봇!")

---
## ✅ Day 1 체크리스트

| # | 항목 | 완료 |
|---|------|------|
| 1 | EXAONE 3.5 모델 로드 성공 | ☐ |
| 2 | Step 1: 시스템 프롬프트 출력 성공 | ☐ |
| 3 | Step 2: safe_generate 테스트 성공 | ☐ |
| 4 | Step 3: 2턴 대화 → 맥락 참조 확인 | ☐ |
| 5 | Step 4: 챗봇 실행 → quit 종료 성공 | ☐ |
| 6 | Step 5: 환각 테스트 (민법 제9999조) | ☐ |
| 7 | Step 6: 품질 불일치 3건 수집 완료 | ☐ |
| 8 | Step 7: 저장 완료 | ☐ |

In [ ]:
print("🎉 Day 1 수고하셨습니다!")
print()
print("오늘의 성과:")
print("  ✓ EXAONE 3.5 2.4B 로컬 추론으로 법률 챗봇 v0.1 완성")
print("  ✓ API 비용 ₩0 — 오픈소스 모델의 비용 이점 체험")
print("  ✓ 품질 불일치 사례 수집 → Day 2 입력 데이터 확보")
print()
print("내일 Day 2:")
print("  ① 품질 불일치 → Zero-shot / Few-shot / CoT 로 개선")
print("  ② LLM-as-Judge — EXAONE이 EXAONE 응답을 자동 평가")
print("  ③ 합성 데이터 생성 — 파인튜닝용 Q&A 자동 제작")
print()
print("💡 오늘 만든 챗봇이 Day 5에서 웹으로 배포됩니다!")